In [1]:

import matplotlib.pyplot as plt
import time
import torch
import numpy as np
from sympy.codegen.rewriting import Optimization
from tqdm import tqdm
from tqdm.notebook import tqdm
import pandas as pd
import torch
import os
from datetime import datetime
import sys
import json
from matplotlib.ticker import LogLocator, LogFormatter
from tqdm import trange
from functools import partial

In [2]:
!pip install flow_matching -q
!pip install POT -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.4 MB/s eta 0:00:00


In [ ]:
import os
import sys
from huggingface_hub import login
from google.colab import userdata
import wandb

hf_token = userdata.get('HF')
if hf_token:
    login(token=hf_token)
else:
    login()

if 'google.colab' in str(get_ipython()):
    import getpass

    !pip install -q diffusers transformers accelerate xformers
    !pip install -q scikit-learn matplotlib Pillow

    github_token = userdata.get('GITHUB')
    if github_token:
        token = github_token
    else:
        token = getpass.getpass("Enter your GitHub personal access token: ")

    repo_url  = f"https://{token}@github.com/orineo1/conditional-matching-paper.git"
    repo_name = "conditional-matching-paper"
    branch    = "adding-simu-compare"

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        print(f"✅ Repo '{repo_name}' already cloned — pulling latest...")
        !cd {repo_name} && git pull

    !cd {repo_name} && git checkout {branch}

    repo_path = f"/content/{repo_name}"
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)

    print(f"\n✅ Repo ready. Branch: {branch}")
    print(f"📁 Python path: {repo_path}")

repo_path = f"/content/{repo_name}/simulations"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.4 MB/s eta 0:00:00


In [ ]:
import importlib
import Diffusion
import superposition_utils

import LossFunctions
import ConsistencyModels
import  FlowMatching
from ConsistencyModels import ConsistencyModel,ConsistencyModeliCT
import dist_utils
import Optimization
import EncoderDecoder
import evalModels

importlib.reload(Diffusion)
importlib.reload(LossFunctions)

importlib.reload(ConsistencyModels)
importlib.reload(FlowMatching)
importlib.reload(dist_utils)
importlib.reload(Optimization)
importlib.reload(superposition_utils)
importlib.reload(EncoderDecoder)
importlib.reload(evalModels)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2D cond on 1D

In [ ]:
mu_list = [torch.tensor([-5,5],dtype=torch.float64),
           torch.tensor([-5,-5],dtype=torch.float64),
           torch.tensor([5, 3],dtype=torch.float64),
           torch.tensor([5,-1],dtype=torch.float64),
           torch.tensor([0, -3],dtype=torch.float64),
           torch.tensor([-2,4 ],dtype=torch.float64),
           torch.tensor([-2,-3 ],dtype=torch.float64),
           torch.tensor([ 1,2],dtype=torch.float64),
           torch.tensor([-7,1],dtype=torch.float64),
           torch.tensor([7,5],dtype=torch.float64),
           torch.tensor([0,-5],dtype=torch.float64)
           ]

Sigma_list = [
    torch.tensor([[0.5000, 0.1950],
                  [0.1950, 0.2000]], dtype=torch.float64)
              ] * len(mu_list)

# Mixture weights
alpha =torch.tensor( [1 / len(mu_list)] * len(mu_list),dtype=torch.float64)

mu_list = [mu.float() for mu in mu_list]
Sigma_list = [cov.float() for cov in Sigma_list]
alpha = alpha.float()

## Target
x_star=torch.tensor([-5])
mu_temp, Sigma_temp =dist_utils.compute_conditionals(mu_list, Sigma_list, x_star)
temp_alpha = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_star)
mog_means, mog_variances, weights= dist_utils.filter_and_normalize(mu_temp, Sigma_temp, temp_alpha, threshold=0.01)


In [ ]:
X = dist_utils.generate_mog_samples(25_000, mu_list, Sigma_list, alpha).float().to(device)
xh_cpu = X.detach().cpu().numpy()

# Scatter plot of first column vs. second column
plt.scatter(xh_cpu[:, 0], xh_cpu[:, 1], alpha=0.6, s=20)
plt.title("Scatter Plot of P(X,Y)")
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)
plt.show()

## Parameters for tests

In [8]:
## NN
nblocks=3
nunits=128
nepochs=20_000
batch_size=512

nepochs_CM=7_500
batch_size_CM=1024

## Diffusion
diffusion_steps=100

## Optimization
n_attemp_optim=25
nsamples_in_optim_for_mmd=250


# For models
condition_on=1
nfeatures= X.shape[1]

## Train

### Consistency Models

Train conditional model  - P(Y|X=x)

In [9]:
B, C = X.shape
nfeatures = C - condition_on

# Use the converted data
data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(nfeatures=nfeatures, condition_on=condition_on, nunits=nunits,depth=nblocks)
Cos_ConsistencyModeliCT.train_model(
    X=None,
    nepochs=nepochs_CM,
    batch_size=batch_size_CM,
    device=device,
    condition=condition_on,
    data_generator=data_generator,
    use_improved_training=True
)

loss: 0.325236, mu: 0.0000, N: 1281: 100%|██████████| 7500/7500 [01:09<00:00, 108.54it/s]


### Diffusion

Train conditional model P(Y|X=x)

In [10]:
## LGD
# init a model, train
X = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)
nfeatures = X.shape[1]
condition_on = mu_list[0].shape[0] - mog_means[0].shape[0]
model_cond = Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=True,
                                      condition_on=condition_on, diffusion_steps=diffusion_steps)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
losses = model_cond.train_model(None,
                                data_generator=data_generator,
                                nepochs=nepochs, batch_size=batch_size, condition_on=condition_on)

loss: 0.281337: 100%|██████████| 20000/20000 [07:48<00:00, 42.67it/s]


Train unconditional model P(X=x)



In [11]:
# init a model, train
X_for_cond_only = X[:, :model_cond.condition_on]
nfeatures = X_for_cond_only.shape[1]

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :model_cond.condition_on])
model_uncond=Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=False,diffusion_steps=diffusion_steps)
losses = model_uncond.train_model(None,
                                  data_generator=data_generator,
                                  nepochs=nepochs
, batch_size=batch_size, condition_on=condition_on)


loss: 0.701443: 100%|██████████| 20000/20000 [07:10<00:00, 46.41it/s]


### Flow

In [12]:
input_dim =mog_means[0].shape[0]
condition_on = mu_list[0].shape[0]-mog_means[0].shape[0]
hidden_dim = nunits
depth = nblocks

Train conditional model P(Y|X=x)



In [13]:
vf_y_cond_x = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=condition_on,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
vf_y_cond_x.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,data_generator=data_generator)



loss: 3.811953: 100%|██████████| 20000/20000 [04:29<00:00, 74.14it/s]


Train unconditional model P(X=x)



In [14]:
input_dim = mu_list[0].shape[0]-mog_means[0].shape[0]#first_column_x.shape[1]
vf_X = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=0,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)
data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :condition_on])

vf_X.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,
              data_generator=data_generator
              )



loss: 6.688751: 100%|██████████| 20000/20000 [02:53<00:00, 115.18it/s]


## Optimize

In [1]:
def gmm_l2_distance(mu_p, Sigma_p, w_p, mu_q, Sigma_q, w_q):
    """
    Exact L2 distance between two GMMs:
    ||p - q||^2 = <p,p> - 2<p,q> + <q,q>
    where <f,g> = integral f(x)g(x)dx, closed form for Gaussians.
    """
    import torch

    def gaussian_inner_product(mu1, S1, w1_list, mu2, S2, w2_list):
        # sum_{i,j} w1_i * w2_j * N(mu1_i; mu2_j, S1_i + S2_j)
        total = 0.0
        for mu_i, S_i, w_i in zip(mu1, S1, w1_list):
            for mu_j, S_j, w_j in zip(mu2, S2, w2_list):
                S_sum = S_i + S_j
                diff  = mu_i - mu_j
                d     = mu_i.shape[0]
                sign, logdet = torch.linalg.slogdet(S_sum)
                log_val = -0.5 * (d * torch.log(torch.tensor(2 * 3.14159265)) + logdet
                                  + diff @ torch.linalg.inv(S_sum) @ diff)
                total += w_i.item() * w_j.item() * torch.exp(log_val).item()
        return total

    pp = gaussian_inner_product(mu_p, Sigma_p, w_p, mu_p, Sigma_p, w_p)
    qq = gaussian_inner_product(mu_q, Sigma_q, w_q, mu_q, Sigma_q, w_q)
    pq = gaussian_inner_product(mu_p, Sigma_p, w_p, mu_q, Sigma_q, w_q)
    return pp - 2 * pq + qq

### LGD

In [16]:
best_x_t_LGD_list = []
l2_gmm_LGD_list   = []
l2_x_LGD_list     = []
lgd_times         = []
final_loss_LGD    = []

for i in trange(n_attemp_optim):
    torch.manual_seed(i)
    np.random.seed(i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(i)
    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, model_cond, mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device)
    end_time = time.time()
    lgd_times.append(end_time - start_time)
    final_loss_LGD.append(final_loss)
    best_x_t_LGD_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_list.append(l2_gmm)
    l2_x_LGD_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")
print(best_x_t_LGD_list)

  4%|▍         | 1/25 [02:12<52:58, 132.42s/it]

[1] L2 GMM: 0.008870  |  L2 to x*: 0.190585


  8%|▊         | 2/25 [04:28<51:39, 134.77s/it]

[2] L2 GMM: 0.026747  |  L2 to x*: 0.332958


 12%|█▏        | 3/25 [06:43<49:19, 134.52s/it]

...

[23] L2 GMM: 0.799220  |  L2 to x*: 9.644735


 96%|█████████▌| 24/25 [53:38<02:13, 133.34s/it]

[24] L2 GMM: 0.000759  |  L2 to x*: 0.050178


100%|██████████| 25/25 [55:50<00:00, 134.03s/it]

[25] L2 GMM: 0.002704  |  L2 to x*: 0.101298
[tensor([[-4.8094]], device='cuda:0'), tensor([[-4.6670]], device='cuda:0'), tensor([[-4.0214]], device='cuda:0'), tensor([[-4.9808]], device='cuda:0'), tensor([[-4.9802]], device='cuda:0'), tensor([[-5.0393]], device='cuda:0'), tensor([[-4.9684]], device='cuda:0'), tensor([[-5.0466]], device='cuda:0'), tensor([[6.3358]], device='cuda:0'), tensor([[-4.9458]], device='cuda:0'), tensor([[-5.0057]], device='cuda:0'), tensor([[-5.0939]], device='cuda:0'), tensor([[-5.0069]], device='cuda:0'), tensor([[3.9206]], device='cuda:0'), tensor([[5.9246]], device='cuda:0'), tensor([[-4.8538]], device='cuda:0'), tensor([[-4.9145]], device='cuda:0'), tensor([[-3.9901]], device='cuda:0'), tensor([[-5.1015]], device='cuda:0'), tensor([[-4.9481]], device='cuda:0'), tensor([[-5.0302]], device='cuda:0'), tensor([[-5.0691]], device='cuda:0'), tensor([[4.6447]], device='cuda:0'), tensor([[-5.0502]], device='cuda:0'), tensor([[-5.1013]], device='cuda:0')]


### LGD-CM

Run the optimization of the LGD-CM

In [17]:
best_x_t_LGD_CM_list = []
l2_gmm_LGD_CM_list   = []
l2_x_LGD_CM_list     = []
lgd_cm_times         = []
final_loss_LGD_CM    = []

for i in trange(n_attemp_optim):
    torch.manual_seed(i)
    np.random.seed(i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(i)
    start_time = time.time()

    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, Cos_ConsistencyModeliCT, mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device,
        CM=True, FLAG=False, num_x_t=3)
    end_time = time.time()
    lgd_cm_times.append(end_time - start_time)
    final_loss_LGD_CM.append(final_loss)
    best_x_t_LGD_CM_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_CM_list.append(l2_gmm)
    l2_x_LGD_CM_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")
print(best_x_t_LGD_CM_list)

  4%|▍         | 1/25 [00:16<06:31, 16.31s/it]

[1] L2 GMM: 0.044900  |  L2 to x*: 0.433881


  8%|▊         | 2/25 [00:32<06:14, 16.26s/it]

[2] L2 GMM: 0.105102  |  L2 to x*: 0.678122


 12%|█▏        | 3/25 [00:49<06:04, 16.56s/it]

...

[23] L2 GMM: 0.084783  |  L2 to x*: 0.604429


 96%|█████████▌| 24/25 [06:30<00:16, 16.37s/it]

[24] L2 GMM: 0.052263  |  L2 to x*: 0.469235


100%|██████████| 25/25 [06:46<00:00, 16.26s/it]

[25] L2 GMM: 0.079108  |  L2 to x*: 0.582658
[tensor([[-4.5661]], device='cuda:0'), tensor([[-4.3219]], device='cuda:0'), tensor([[-4.4826]], device='cuda:0'), tensor([[-4.5868]], device='cuda:0'), tensor([[-4.4678]], device='cuda:0'), tensor([[-4.3674]], device='cuda:0'), tensor([[-4.7298]], device='cuda:0'), tensor([[-4.3549]], device='cuda:0'), tensor([[-4.4928]], device='cuda:0'), tensor([[-4.4271]], device='cuda:0'), tensor([[-4.4740]], device='cuda:0'), tensor([[-4.5161]], device='cuda:0'), tensor([[-4.5224]], device='cuda:0'), tensor([[-4.2888]], device='cuda:0'), tensor([[-4.5082]], device='cuda:0'), tensor([[-4.4807]], device='cuda:0'), tensor([[-4.4155]], device='cuda:0'), tensor([[-4.7105]], device='cuda:0'), tensor([[-4.5868]], device='cuda:0'), tensor([[-4.3312]], device='cuda:0'), tensor([[-4.5670]], device='cuda:0'), tensor([[-4.3828]], device='cuda:0'), tensor([[-4.3956]], device='cuda:0'), tensor([[-4.5308]], device='cuda:0'), tensor([[-4.4173]], device='cuda:0')]


### D-FLOW

In [18]:
x_optim_dflow_list    = []
l2_gmm_dflow_list     = []
l2_x_dflow_list       = []
dflow_times           = []
final_loss_dflow_list = []

for i in trange(n_attemp_optim):
    torch.manual_seed(i)
    np.random.seed(i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(i)
    start_time = time.time()
    x_optim, final_loss = Optimization.optimize_DFLOW(
        vf_y_cond_x, vf_X, device, mog_means, mog_variances, weights,
        max_iter=100, FLAG=False, n_sample=nsamples_in_optim_for_mmd,
        loss_method="MMD", line_search_fn="strong_wolfe")
    end_time = time.time()
    dflow_times.append(end_time - start_time)
    final_loss_dflow_list.append(final_loss)
    x_optim_dflow_list.append(x_optim)

    x_pred_t = x_optim.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_dflow_list.append(l2_gmm)
    l2_x_dflow_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")
print(x_optim_dflow_list)

  4%|▍         | 1/25 [00:01<00:38,  1.62s/it]

[1] L2 GMM: 0.008814  |  L2 to x*: 0.189987


  8%|▊         | 2/25 [00:07<01:28,  3.84s/it]

[2] L2 GMM: 0.613870  |  L2 to x*: 3.486731


 12%|█▏        | 3/25 [00:09<01:14,  3.38s/it]

...

[23] L2 GMM: 0.398699  |  L2 to x*: 1.471226


 96%|█████████▌| 24/25 [02:29<00:13, 13.65s/it]

[24] L2 GMM: 0.365934  |  L2 to x*: 11.223812


100%|██████████| 25/25 [02:35<00:00,  6.21s/it]

[25] L2 GMM: 0.407068  |  L2 to x*: 5.446252
[tensor([[-4.8100]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-1.5133]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.9997]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[0.7341]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.0720]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[0.6500]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-1.3090]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-1.3933]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.9885]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.4033]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.9983]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-5.3730]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.7677]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[6.1864]], device='cuda:0', grad_fn=<SelectBackward0>), tensor([[-4.8942]], device='cuda:0', grad_fn=<S

In [26]:

def summary_row(name, l2_gmm, l2_x, times):
    return {
        "Method":               name,
        "L2 GMM mean":          f"{np.mean(l2_gmm):.4f}",
        "L2 GMM std":           f"{np.std(l2_gmm):.4f}",
        "L2 to x* mean":        f"{np.mean(l2_x):.4f}",
        "L2 to x* std":         f"{np.std(l2_x):.4f}",
        "Time mean (s)":        f"{np.mean(times):.2f}",
        "Time std (s)":         f"{np.std(times):.2f}",
    }


def top10_stats(name, final_loss, l2_gmm, l2_x, times):
    losses = [fl.item() if hasattr(fl, 'item') else fl for fl in final_loss]
    k = min(10, len(losses))
    top10_idx = np.argsort(losses)[:k]

    top10_loss = [losses[i]    for i in top10_idx]
    top10_gmm  = [l2_gmm[i]   for i in top10_idx]
    top10_x    = [l2_x[i]     for i in top10_idx]
    top10_time = [times[i]     for i in top10_idx]

    return {
        "Method":            name,
        "Loss mean":         f"{np.mean(top10_loss):.4f}",
        "Loss std":          f"{np.std(top10_loss):.4f}",
        "L2 GMM mean":       f"{np.mean(top10_gmm):.4f}",
        "L2 GMM std":        f"{np.std(top10_gmm):.4f}",
        "L2 to x* mean":     f"{np.mean(top10_x):.4f}",
        "L2 to x* std":      f"{np.std(top10_x):.4f}",
        "Time mean (s)":     f"{np.mean(top10_time):.2f}",
        "Time std (s)":      f"{np.std(top10_time):.2f}",
        "Top-k selected":    k,
    }


In [27]:

rows = [
    summary_row("LGD",     l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    summary_row("LGD-CM",  l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
    summary_row("D-Flow",  l2_gmm_dflow_list,  l2_x_dflow_list,  dflow_times),
]

df = pd.DataFrame(rows).set_index("Method")
display(df)

rows = [
    top10_stats("LGD",    final_loss_LGD,    l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    top10_stats("LGD-CM", final_loss_LGD_CM, l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
    top10_stats("D-Flow", final_loss_dflow_list, l2_gmm_dflow_list, l2_x_dflow_list, dflow_times),
]

df = pd.DataFrame(rows).set_index("Method")
display(df)


,L2 GMM mean,L2 GMM std,L2 to x* mean,L2 to x* std,Time mean (s),Time std (s)
Method,,,,,,
LGD,0.1174,0.2379,1.7716,3.7104,133.99,1.78
LGD-CM,0.0666,0.0247,0.5230,0.1094,16.22,0.40
D-Flow,0.2247,0.2451,2.4424,3.2422,6.16,7.23


,Loss mean,Loss std,L2 GMM mean,L2 GMM std,L2 to x* mean,L2 to x* std,Time mean (s),Time std (s),Top-k selected
Method,,,,,,,,,
LGD,0.0015,0.0010,0.0036,0.0078,0.0805,0.0898,134.12,1.71,10
LGD-CM,0.0047,0.0017,0.0738,0.0194,0.5575,0.0768,16.41,0.31,10
D-Flow,0.0091,0.0058,0.0140,0.0169,0.1735,0.1659,3.64,1.52,10


In [29]:
import json

def to_python(val):
    """Convert tensors/numpy to plain Python for JSON serialization."""
    if isinstance(val, torch.Tensor):
        return val.detach().cpu().tolist()
    if isinstance(val, np.ndarray):
        return val.tolist()
    if hasattr(val, 'item'):
        return val.item()
    return val

results = {
    "LGD": {
        "x_pred":     [to_python(x) for x in best_x_t_LGD_list],
        "final_loss": [to_python(l) for l in final_loss_LGD],
        "l2_gmm":     l2_gmm_LGD_list,
        "l2_x":       l2_x_LGD_list,
        "times":      lgd_times,
    },
    "LGD-CM": {
        "x_pred":     [to_python(x) for x in best_x_t_LGD_CM_list],
        "final_loss": [to_python(l) for l in final_loss_LGD_CM],
        "l2_gmm":     l2_gmm_LGD_CM_list,
        "l2_x":       l2_x_LGD_CM_list,
        "times":      lgd_cm_times,
    },
    "D-Flow": {
        "x_pred":     [to_python(x) for x in x_optim_dflow_list],
        "final_loss": [to_python(l) for l in final_loss_dflow_list],
        "l2_gmm":     l2_gmm_dflow_list,
        "l2_x":       l2_x_dflow_list,
        "times":      dflow_times,
    },
    "meta": {
        "n_attemp_optim":          n_attemp_optim,
        "nsamples_in_optim_for_mmd": nsamples_in_optim_for_mmd,
        "x_star":                  to_python(x_star),
    }
}

# save
with open("results2d.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

{
  "LGD": {
    "x_pred": [
      [
        [
          -4.809414863586426
        ]
      ],
      [
        [
          -4.667042255401611
        ]
      ],
      [
        [
          -4.021422386169434
        ]
      ],
      [
        [
          -4.9807868003845215
        ]
      ],
      [
        [
          -4.980242729187012
        ]
      ],
      [
        [
          -5.039294242858887
        ]
      ],
      [
        [
          -4.968446731567383
        ]
      ],
      [
        [
          -5.0465545654296875
        ]
      ],
      [
        [
          6.3358049392700195
        ]
      ],
      [
        [
          -4.945843696594238
        ]
      ],
      [
        [
          -5.005720615386963
        ]
      ],
      [
        [
          -5.093913555145264
        ]
      ],
      [
        [
          -5.006936550140381
        ]
      ],
      [
        [
          3.9206297397613525
        ]
      ],
      [
        [
          5.924609184265137

## Summary - to 10

# 10D cond on 1D

## Parameters for tests

In [62]:
## NN
nblocks=6
nunits=128
nepochs=20_000
batch_size=512

nepochs_CM=30_000
batch_size_CM=4096

## Diffusion
diffusion_steps=100

## Optimization
n_attemp_optim=25
nsamples_in_optim_for_mmd=250

In [63]:
mu_list, Sigma_list, alpha,mog_means, mog_variances,weights,x_star= dist_utils.get_param_mog_with_target(dim_data=10,num_components=4,device='cpu',conditional_modes=2,distanceOrScale="Distance")
mog_means, mog_variances, weights= dist_utils.filter_and_normalize(mog_means, mog_variances, weights, threshold=0.001)

## Train

### Consistency Models

Train conditional model  - P(Y|X=x)

In [ ]:
# init a model, train
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

X = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)

condition_on=9
B, C = X.shape
nfeatures = C - condition_on
mu_list = [mu.float() for mu in mu_list]
Sigma_list = [cov.float() for cov in Sigma_list]
alpha = alpha.float()

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(nfeatures=nfeatures, condition_on=condition_on, nunits=nunits,depth=nblocks)
Cos_ConsistencyModeliCT.train_model(X=None, nepochs=nepochs_CM
                      ,batch_size=batch_size_CM, device= device, condition=condition_on,
                      data_generator=data_generator,
                      )

loss: 0.629184, mu: 0.0000, N: 21:  23%|██▎       | 6758/30000 [00:51<02:52, 134.44it/s]

### Diffusion

Train conditional model P(Y|X=x)

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# init a model, train
X = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)
nfeatures = X.shape[1]
condition_on = mu_list[0].shape[0] - mog_means[0].shape[0]
model_cond = Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=True,
                                      condition_on=condition_on, diffusion_steps=diffusion_steps)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
losses = model_cond.train_model(None,
                                data_generator=data_generator,
                                nepochs=nepochs, batch_size=batch_size, condition_on=condition_on)

Train uncnditional model P(X=x)

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# init a model, train
X_for_cond_only = X[:, :model_cond.condition_on]
nfeatures = X_for_cond_only.shape[1]

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :model_cond.condition_on])
model_uncond=Diffusion.DiffusionModel(nfeatures=nfeatures, nblocks=nblocks, nunits=nunits, condition=False,diffusion_steps=diffusion_steps)
losses = model_uncond.train_model(None,
                                  data_generator=data_generator,
                                  nepochs=nepochs
, batch_size=batch_size, condition_on=condition_on)


### Flow

In [ ]:
input_dim =mog_means[0].shape[0]
y_dim = mu_list[0].shape[0]-mog_means[0].shape[0]
hidden_dim = nunits
depth = nblocks

Train conditional model P(Y|X=x)

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

vf_y_cond_x = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=condition_on,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)

data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=None)
vf_y_cond_x.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,data_generator=data_generator)



Train conditional model P(X=x)

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

input_dim = mu_list[0].shape[0]-mog_means[0].shape[0]#first_column_x.shape[1]
vf_X = FlowMatching.FMModel(nfeatures=input_dim,
                                   condition_on=0,
                                   nunits=hidden_dim,
                                   nblocks=depth,
                                   device=device)
data_generator=partial(dist_utils.generate_mog_samples_not_differentiable, means=mu_list, variances=Sigma_list, weights=alpha,kernel_func=lambda X: X[:, :condition_on])

vf_X.train_FM(lr=0.001, batch_size=batch_size, nepochs=nepochs,
              data_generator=data_generator
              )

## Optimize

### LGD

In [ ]:
best_x_t_LGD_list = []
l2_gmm_LGD_list   = []
l2_x_LGD_list     = []
lgd_times         = []
final_loss_LGD    = []
for i in trange(n_attemp_optim):
    torch.manual_seed(i)
    np.random.seed(i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(i)
    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, model_cond, mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device)
    best_x_t = best_x_t.reshape(-1, 1)
    end_time = time.time()
    lgd_times.append(end_time - start_time)
    final_loss_LGD.append(final_loss)
    best_x_t_LGD_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_list.append(l2_gmm)
    l2_x_LGD_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")



In [ ]:
#  4%|▍         | 1/25 [03:07<1:14:50, 187.11s/it][1] L2 GMM: 0.132829  |  L2 to x*: 23.632969
#   8%|▊         | 2/25 [06:13<1:11:34, 186.72s/it][2] L2 GMM: 0.287109  |  L2 to x*: 21.547514
#  12%|█▏        | 3/25 [09:22<1:08:47, 187.60s/it][3] L2 GMM: 0.317680  |  L2 to x*: 17.913181
#  16%|█▌        | 4/25 [12:28<1:05:27, 187.03s/it][4] L2 GMM: 0.161391  |  L2 to x*: 25.615641
#  20%|██        | 5/25 [15:28<1:01:29, 184.49s/it][5] L2 GMM: 0.097503  |  L2 to x*: 5.921921
#  24%|██▍       | 6/25 [18:33<58:29, 184.70s/it]  [6] L2 GMM: 0.053769  |  L2 to x*: 5.059596
#  28%|██▊       | 7/25 [21:48<56:23, 187.95s/it][7] L2 GMM: 0.723154  |  L2 to x*: 12.182523
#  32%|███▏      | 8/25 [25:06<54:13, 191.39s/it][8] L2 GMM: 0.164866  |  L2 to x*: 6.245852
#  36%|███▌      | 9/25 [28:14<50:43, 190.20s/it][9] L2 GMM: 0.231161  |  L2 to x*: 8.652859
#  40%|████      | 10/25 [31:22<47:24, 189.65s/it][10] L2 GMM: 0.130572  |  L2 to x*: 18.498243
#  44%|████▍     | 11/25 [34:31<44:08, 189.21s/it][11] L2 GMM: 0.371486  |  L2 to x*: 6.874916
#  48%|████▊     | 12/25 [37:38<40:53, 188.76s/it][12] L2 GMM: 0.310869  |  L2 to x*: 18.459089
#  52%|█████▏    | 13/25 [40:47<37:43, 188.65s/it][13] L2 GMM: 0.723164  |  L2 to x*: 16.971172
#  56%|█████▌    | 14/25 [43:55<34:32, 188.42s/it][14] L2 GMM: 0.203027  |  L2 to x*: 14.025564
#  60%|██████    | 15/25 [47:03<31:25, 188.54s/it][15] L2 GMM: 0.271959  |  L2 to x*: 42.722347
#  64%|██████▍   | 16/25 [50:14<28:22, 189.22s/it][16] L2 GMM: 0.318351  |  L2 to x*: 11.252840
#  68%|██████▊   | 17/25 [53:24<25:15, 189.45s/it][17] L2 GMM: 0.294222  |  L2 to x*: 16.088886
#  72%|███████▏  | 18/25 [56:36<22:11, 190.25s/it][18] L2 GMM: 0.503868  |  L2 to x*: 7.389486
#  76%|███████▌  | 19/25 [59:46<19:00, 190.15s/it][19] L2 GMM: 0.841668  |  L2 to x*: 13.128753
#  80%|████████  | 20/25 [1:02:52<15:44, 188.84s/it][20] L2 GMM: 0.115661  |  L2 to x*: 44.453335
#  84%|████████▍ | 21/25 [1:05:58<12:32, 188.09s/it][21] L2 GMM: 0.308995  |  L2 to x*: 17.123634
#  88%|████████▊ | 22/25 [1:09:04<09:22, 187.34s/it][22] L2 GMM: 0.800741  |  L2 to x*: 6.413753
#  92%|█████████▏| 23/25 [1:12:08<06:12, 186.45s/it][23] L2 GMM: 0.431512  |  L2 to x*: 5.439972
#  96%|█████████▌| 24/25 [1:15:14<03:06, 186.11s/it][24] L2 GMM: 0.632809  |  L2 to x*: 24.905619
# 100%|██████████| 25/25 [1:18:19<00:00, 187.96s/it][25] L2 GMM: 0.499136  |  L2 to x*: 10.158223


### LGD-CM

In [ ]:
best_x_t_LGD_CM_list = []
l2_gmm_LGD_CM_list   = []
l2_x_LGD_CM_list     = []
lgd_cm_times         = []
final_loss_LGD_CM    = []

for i in trange(n_attemp_optim):
    torch.manual_seed(i)
    np.random.seed(i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(i)
    start_time = time.time()
    best_x_t, best_x_0_cont_xt_hat, final_loss = Optimization.optimize_LGD(
        model_uncond, Cos_ConsistencyModeliCT, mog_means, mog_variances, weights,
        mu_list, Sigma_list, alpha,
        nsamples=nsamples_in_optim_for_mmd, loss="MMD", device=device,
        CM=True, FLAG=False, num_x_t=10)
    best_x_t = best_x_t.reshape(-1, 1)
    end_time = time.time()
    lgd_cm_times.append(end_time - start_time)
    final_loss_LGD_CM.append(final_loss)
    best_x_t_LGD_CM_list.append(best_x_t)

    x_pred_t = best_x_t.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_LGD_CM_list.append(l2_gmm)
    l2_x_LGD_CM_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")

print(best_x_t_LGD_CM_list)

In [ ]:
np.array(l2_gmm_LGD_CM_list)[np.argsort(final_loss_LGD_CM)[:10]]

In [ ]:
k = min(10, len(final_loss_LGD_CM))
top10_idx = np.argsort(final_loss_LGD_CM)[:k]
second_idx = top10_idx[int(np.argsort([l2_gmm_LGD_CM_list[i] for i in top10_idx])[1])]

x_optimized = best_x_t_LGD_CM_list[second_idx].float().view(-1).cpu()

mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_optimized)
w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_optimized)
mu_pred, Sigma_pred, w_pred = dist_utils.filter_and_normalize(
    mu_pred, Sigma_pred, w_pred, threshold=0.01)

fig, ax = plt.subplots(figsize=(10, 5))

plot_gmm_1d(mog_means, mog_variances, weights,
            label="Target $p(y|x^*)$", color="steelblue", ax=ax)

plot_gmm_1d(mu_pred, Sigma_pred, w_pred,
            label=f"LGD-CM 2nd best (seed={second_idx}, loss={final_loss_LGD_CM[second_idx]:.4f}, L2 GMM={l2_gmm_LGD_CM_list[second_idx]:.4f})",
            color="seagreen", ax=ax, linestyle='--')

ax.set_xlabel("y")
ax.set_ylabel("Density")
ax.set_title("Target vs LGD-CM 2nd best predicted conditional distribution")
ax.legend()
plt.tight_layout()
plt.show()

### D-FLOW

In [ ]:
x_optim_dflow_list    = []
l2_gmm_dflow_list     = []
l2_x_dflow_list       = []
dflow_times           = []
final_loss_dflow_list = []

for i in trange(n_attemp_optim):
    torch.manual_seed(i)
    np.random.seed(i)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(i)
    start_time = time.time()
    x_optim, final_loss = Optimization.optimize_DFLOW(
        vf_y_cond_x, vf_X, device, mog_means, mog_variances, weights,
        max_iter=100, FLAG=False, n_sample=nsamples_in_optim_for_mmd,
        loss_method="MMD", line_search_fn="strong_wolfe")
    x_optim = x_optim.reshape(-1, 1)
    end_time = time.time()
    dflow_times.append(end_time - start_time)
    final_loss_dflow_list.append(final_loss)
    x_optim_dflow_list.append(x_optim)

    x_pred_t = x_optim.float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)

    l2_gmm = gmm_l2_distance(mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights)
    l2_x   = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

    l2_gmm_dflow_list.append(l2_gmm)
    l2_x_dflow_list.append(l2_x)
    print(f"[{i+1}] L2 GMM: {l2_gmm:.6f}  |  L2 to x*: {l2_x:.6f}")

print(x_optim_dflow_list)

In [ ]:

rows = [
    summary_row("LGD",     l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    summary_row("LGD-CM",  l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
    summary_row("D-Flow",  l2_gmm_dflow_list,  l2_x_dflow_list,  dflow_times),
]

df = pd.DataFrame(rows).set_index("Method")
display(df)

rows = [
    top10_stats("LGD",    final_loss_LGD,    l2_gmm_LGD_list,    l2_x_LGD_list,    lgd_times),
    top10_stats("LGD-CM", final_loss_LGD_CM, l2_gmm_LGD_CM_list, l2_x_LGD_CM_list, lgd_cm_times),
    top10_stats("D-Flow", final_loss_dflow_list, l2_gmm_dflow_list, l2_x_dflow_list, dflow_times),
]

df = pd.DataFrame(rows).set_index("Method")
display(df)


In [ ]:

results = {
    "LGD": {
        "x_pred":     [to_python(x) for x in best_x_t_LGD_list],
        "final_loss": [to_python(l) for l in final_loss_LGD],
        "l2_gmm":     l2_gmm_LGD_list,
        "l2_x":       l2_x_LGD_list,
        "times":      lgd_times,
    },
    "LGD-CM": {
        "x_pred":     [to_python(x) for x in best_x_t_LGD_CM_list],
        "final_loss": [to_python(l) for l in final_loss_LGD_CM],
        "l2_gmm":     l2_gmm_LGD_CM_list,
        "l2_x":       l2_x_LGD_CM_list,
        "times":      lgd_cm_times,
    },
    "D-Flow": {
        "x_pred":     [to_python(x) for x in x_optim_dflow_list],
        "final_loss": [to_python(l) for l in final_loss_dflow_list],
        "l2_gmm":     l2_gmm_dflow_list,
        "l2_x":       l2_x_dflow_list,
        "times":      dflow_times,
    },
    "meta": {
        "n_attemp_optim":          n_attemp_optim,
        "nsamples_in_optim_for_mmd": nsamples_in_optim_for_mmd,
        "x_star":                  to_python(x_star),
    }
}

# save
with open("results10d.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

In [ ]:

# ── plot function ─────────────────────────────────────────────────────────────
def plot_gmm_1d(means, variances, weights, label, color, ax, alpha=0.6, linestyle='-'):
    if isinstance(means, torch.Tensor):
        means_list = [means[i] for i in range(means.shape[0])]
    else:
        means_list = means

    if isinstance(variances, torch.Tensor):
        vars_list = [variances[i] for i in range(variances.shape[0])]
    else:
        vars_list = variances

    weights_np = weights.cpu().numpy() if isinstance(weights, torch.Tensor) else np.array(weights)

    all_means = np.array([m.cpu().numpy().flatten()[0] for m in means_list])
    all_stds  = np.array([
        v.sqrt().item() if v.numel() == 1
        else v.diag().sqrt().cpu().numpy()[0]
        for v in vars_list
    ])
    x_min = all_means.min() - 4 * all_stds.max()
    x_max = all_means.max() + 4 * all_stds.max()
    x = np.linspace(x_min, x_max, 1000)

    density = np.zeros_like(x)
    for mu, sigma, w in zip(means_list, vars_list, weights_np):
        mu_val  = mu.cpu().numpy().flatten()[0]
        std_val = (sigma.sqrt().item() if sigma.numel() == 1
                   else sigma.diag().sqrt().cpu().numpy()[0])
        density += w * (1 / (std_val * np.sqrt(2 * np.pi))) * \
                   np.exp(-0.5 * ((x - mu_val) / std_val) ** 2)

    ax.plot(x, density, label=label, color=color, linestyle=linestyle, linewidth=2, alpha=alpha)
    ax.fill_between(x, density, alpha=0.15, color=color)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import torch

# # ── load results ──────────────────────────────────────────────────────────────
# with open("results.json") as f:
#     results = json.load(f)

def load_method(results, method_name):
    d = results[method_name]
    x_list    = [torch.tensor(x) for x in d["x_pred"]]
    loss_list = d["final_loss"]
    l2_gmm    = d["l2_gmm"]
    return x_list, loss_list, l2_gmm

best_x_t_LGD_list,    final_loss_LGD,       l2_gmm_LGD    = load_method(results, "LGD")
best_x_t_LGD_CM_list, final_loss_LGD_CM,    l2_gmm_LGD_CM = load_method(results, "LGD-CM")
x_optim_dflow_list,   final_loss_dflow_list, l2_gmm_dflow  = load_method(results, "D-Flow")

x_star = torch.tensor(results["meta"]["x_star"])



# ── methods & colors ──────────────────────────────────────────────────────────
methods = {
    "LGD":    (best_x_t_LGD_list,    final_loss_LGD,       l2_gmm_LGD),
    "LGD-CM": (best_x_t_LGD_CM_list, final_loss_LGD_CM,    l2_gmm_LGD_CM),
    # "D-Flow": (x_optim_dflow_list,   final_loss_dflow_list, l2_gmm_dflow),
}
colors = {
    "LGD":    "tomato",
    "LGD-CM": "seagreen",
    # "D-Flow": "darkorange",
}

# ── plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

# target p(y | x*)
plot_gmm_1d(mog_means, mog_variances, weights,
            label="Target $p(y|x^*)$", color="steelblue", ax=ax)

for method_name, (x_list, loss_list, l2_gmm_list) in methods.items():
    # top-10 by final loss, then pick lowest l2_gmm among them
    k = min(10, len(loss_list))
    top10_idx = np.argsort(loss_list)[:k]
    best_idx  = top10_idx[int(np.argmin([l2_gmm_list[i] for i in top10_idx]))]

    x_pred_best = x_list[best_idx].float().view(-1).cpu()
    mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_best)
    w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_best)
    mu_pred, Sigma_pred, w_pred = dist_utils.filter_and_normalize(
        mu_pred, Sigma_pred, w_pred, threshold=0.01)

    plot_gmm_1d(mu_pred, Sigma_pred, w_pred,
                label=f"{method_name} (seed={best_idx}, loss={loss_list[best_idx]:.4f}, L2 GMM={l2_gmm_list[best_idx]:.4f})",
                color=colors[method_name], ax=ax, linestyle='--')

ax.set_xlabel("y")
ax.set_ylabel("Density")
ax.set_title("Target vs best predicted conditional distribution (by final loss)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# @title
# # @title
# results={
#   "LGD": {
#     "x_pred": [
#       [
#         [
#           -9.200787544250488
#         ],
#         [
#           2.1044507026672363
#         ],
#         [
#           -8.698792457580566
#         ],
#         [
#           -17.69344139099121
#         ],
#         [
#           -1.0692639350891113
#         ],
#         [
#           -3.3037848472595215
#         ],
#         [
#           19.481857299804688
#         ],
#         [
#           6.700495719909668
#         ],
#         [
#           5.383772373199463
#         ]
#       ],
#       [
#         [
#           -16.029132843017578
#         ],
#         [
#           0.010957539081573486
#         ],
#         [
#           -13.97431468963623
#         ],
#         [
#           -12.779995918273926
#         ],
#         [
#           0.699354887008667
#         ],
#         [
#           -6.372217178344727
#         ],
#         [
#           12.575362205505371
#         ],
#         [
#           0.47914326190948486
#         ],
#         [
#           -3.5209579467773438
#         ]
#       ],
#       [
#         [
#           -16.22937774658203
#         ],
#         [
#           -0.15603995323181152
#         ],
#         [
#           -8.139263153076172
#         ],
#         [
#           -9.393655776977539
#         ],
#         [
#           -0.27963876724243164
#         ],
#         [
#           -3.393833875656128
#         ],
#         [
#           10.872742652893066
#         ],
#         [
#           4.346153259277344
#         ],
#         [
#           -2.500990390777588
#         ]
#       ],
#       [
#         [
#           -2.5753862857818604
#         ],
#         [
#           13.982893943786621
#         ],
#         [
#           -8.136099815368652
#         ],
#         [
#           -20.4289608001709
#         ],
#         [
#           -2.5961642265319824
#         ],
#         [
#           17.361495971679688
#         ],
#         [
#           1.533831000328064
#         ],
#         [
#           -2.601713180541992
#         ],
#         [
#           -4.152698516845703
#         ]
#       ],
#       [
#         [
#           -6.050750255584717
#         ],
#         [
#           3.437685489654541
#         ],
#         [
#           -3.1591567993164062
#         ],
#         [
#           -13.745190620422363
#         ],
#         [
#           3.6444458961486816
#         ],
#         [
#           0.3946475386619568
#         ],
#         [
#           7.237149715423584
#         ],
#         [
#           -1.046478509902954
#         ],
#         [
#           -2.8311767578125
#         ]
#       ],
#       [
#         [
#           -5.8409743309021
#         ],
#         [
#           3.40397572517395
#         ],
#         [
#           -3.0910327434539795
#         ],
#         [
#           -12.791829109191895
#         ],
#         [
#           4.070526599884033
#         ],
#         [
#           1.3470889329910278
#         ],
#         [
#           5.513212203979492
#         ],
#         [
#           -1.1905264854431152
#         ],
#         [
#           -3.0964622497558594
#         ]
#       ],
#       [
#         [
#           -6.945877552032471
#         ],
#         [
#           5.639953136444092
#         ],
#         [
#           -3.040804862976074
#         ],
#         [
#           -7.951323986053467
#         ],
#         [
#           2.252903461456299
#         ],
#         [
#           6.689900875091553
#         ],
#         [
#           -0.698192298412323
#         ],
#         [
#           -6.811567783355713
#         ],
#         [
#           -3.505586862564087
#         ]
#       ],
#       [
#         [
#           -5.283031463623047
#         ],
#         [
#           3.4234671592712402
#         ],
#         [
#           -3.7009711265563965
#         ],
#         [
#           -13.923707962036133
#         ],
#         [
#           4.422874927520752
#         ],
#         [
#           0.8298449516296387
#         ],
#         [
#           6.8578996658325195
#         ],
#         [
#           -0.6656991243362427
#         ],
#         [
#           -2.974412441253662
#         ]
#       ],
#       [
#         [
#           -8.099324226379395
#         ],
#         [
#           2.1975057125091553
#         ],
#         [
#           -3.8700308799743652
#         ],
#         [
#           -7.876715660095215
#         ],
#         [
#           6.596723556518555
#         ],
#         [
#           0.7972033619880676
#         ],
#         [
#           7.650460243225098
#         ],
#         [
#           -0.7008482813835144
#         ],
#         [
#           -2.2011308670043945
#         ]
#       ],
#       [
#         [
#           -14.974632263183594
#         ],
#         [
#           0.1336081475019455
#         ],
#         [
#           -8.64394760131836
#         ],
#         [
#           -11.69769287109375
#         ],
#         [
#           -0.21431206166744232
#         ],
#         [
#           -3.131899833679199
#         ],
#         [
#           12.438326835632324
#         ],
#         [
#           5.304195880889893
#         ],
#         [
#           -0.269187867641449
#         ]
#       ],
#       [
#         [
#           -5.173746109008789
#         ],
#         [
#           2.730191707611084
#         ],
#         [
#           -3.2567296028137207
#         ],
#         [
#           -10.77601146697998
#         ],
#         [
#           3.670961856842041
#         ],
#         [
#           1.4771302938461304
#         ],
#         [
#           7.513754367828369
#         ],
#         [
#           2.140418291091919
#         ],
#         [
#           -1.9446566104888916
#         ]
#       ],
#       [
#         [
#           -2.8836312294006348
#         ],
#         [
#           -3.913753032684326
#         ],
#         [
#           0.7237539291381836
#         ],
#         [
#           -1.5565433502197266
#         ],
#         [
#           -1.7402448654174805
#         ],
#         [
#           5.786696910858154
#         ],
#         [
#           -6.570561408996582
#         ],
#         [
#           -3.5416362285614014
#         ],
#         [
#           -7.77406644821167
#         ]
#       ],
#       [
#         [
#           -0.7047008275985718
#         ],
#         [
#           -5.4187912940979
#         ],
#         [
#           0.9934382438659668
#         ],
#         [
#           -0.8826854228973389
#         ],
#         [
#           1.1031498908996582
#         ],
#         [
#           3.1604833602905273
#         ],
#         [
#           -4.163578510284424
#         ],
#         [
#           -1.0449259281158447
#         ],
#         [
#           -7.300136566162109
#         ]
#       ],
#       [
#         [
#           -11.05102825164795
#         ],
#         [
#           1.4669922590255737
#         ],
#         [
#           -5.6006293296813965
#         ],
#         [
#           -10.24157428741455
#         ],
#         [
#           2.9569828510284424
#         ],
#         [
#           -2.338419198989868
#         ],
#         [
#           13.215585708618164
#         ],
#         [
#           2.911318302154541
#         ],
#         [
#           -0.5808376669883728
#         ]
#       ],
#       [
#         [
#           20.152130126953125
#         ],
#         [
#           -13.589765548706055
#         ],
#         [
#           12.640107154846191
#         ],
#         [
#           1.5063785314559937
#         ],
#         [
#           -1.1312587261199951
#         ],
#         [
#           3.5344161987304688
#         ],
#         [
#           -1.9516974687576294
#         ],
#         [
#           22.466629028320312
#         ],
#         [
#           -1.7115386724472046
#         ]
#       ],
#       [
#         [
#           -9.444230079650879
#         ],
#         [
#           1.3830410242080688
#         ],
#         [
#           -2.9250991344451904
#         ],
#         [
#           -9.05382251739502
#         ],
#         [
#           4.248472213745117
#         ],
#         [
#           -1.202115535736084
#         ],
#         [
#           12.44152545928955
#         ],
#         [
#           1.2116364240646362
#         ],
#         [
#           -1.3127658367156982
#         ]
#       ],
#       [
#         [
#           -8.301036834716797
#         ],
#         [
#           4.400271892547607
#         ],
#         [
#           -2.0797712802886963
#         ],
#         [
#           -14.574814796447754
#         ],
#         [
#           2.259096145629883
#         ],
#         [
#           0.7712579369544983
#         ],
#         [
#           14.458236694335938
#         ],
#         [
#           7.09791898727417
#         ],
#         [
#           2.751897096633911
#         ]
#       ],
#       [
#         [
#           -4.988931655883789
#         ],
#         [
#           0.728408694267273
#         ],
#         [
#           2.0545272827148438
#         ],
#         [
#           -7.670808792114258
#         ],
#         [
#           4.839168548583984
#         ],
#         [
#           1.0923956632614136
#         ],
#         [
#           7.739095211029053
#         ],
#         [
#           2.195777177810669
#         ],
#         [
#           -2.0162770748138428
#         ]
#       ],
#       [
#         [
#           -10.66714859008789
#         ],
#         [
#           0.009976140223443508
#         ],
#         [
#           3.4335503578186035
#         ],
#         [
#           -4.385544776916504
#         ],
#         [
#           0.4893749952316284
#         ],
#         [
#           4.643416404724121
#         ],
#         [
#           0.6576093435287476
#         ],
#         [
#           -5.9774932861328125
#         ],
#         [
#           -5.3770551681518555
#         ]
#       ],
#       [
#         [
#           20.896896362304688
#         ],
#         [
#           -17.164257049560547
#         ],
#         [
#           16.418655395507812
#         ],
#         [
#           -2.987330198287964
#         ],
#         [
#           3.3460142612457275
#         ],
#         [
#           1.4085193872451782
#         ],
#         [
#           -0.45873111486434937
#         ],
#         [
#           22.34882164001465
#         ],
#         [
#           1.5578734874725342
#         ]
#       ],
#       [
#         [
#           -13.70439624786377
#         ],
#         [
#           1.0594974756240845
#         ],
#         [
#           -6.539939880371094
#         ],
#         [
#           -9.952341079711914
#         ],
#         [
#           1.3071256875991821
#         ],
#         [
#           -3.8545379638671875
#         ],
#         [
#           14.578435897827148
#         ],
#         [
#           3.60170578956604
#         ],
#         [
#           -0.9547370672225952
#         ]
#       ],
#       [
#         [
#           -7.182413101196289
#         ],
#         [
#           6.0120744705200195
#         ],
#         [
#           -0.7087766528129578
#         ],
#         [
#           -12.496495246887207
#         ],
#         [
#           2.4289066791534424
#         ],
#         [
#           1.167438268661499
#         ],
#         [
#           6.836133003234863
#         ],
#         [
#           -3.667686700820923
#         ],
#         [
#           -2.192678689956665
#         ]
#       ],
#       [
#         [
#           -4.968361854553223
#         ],
#         [
#           1.869828224182129
#         ],
#         [
#           -1.0303770303726196
#         ],
#         [
#           -8.709364891052246
#         ],
#         [
#           4.6344499588012695
#         ],
#         [
#           0.18870098888874054
#         ],
#         [
#           7.31505823135376
#         ],
#         [
#           0.8630776405334473
#         ],
#         [
#           -2.682631492614746
#         ]
#       ],
#       [
#         [
#           -5.92108678817749
#         ],
#         [
#           16.99488639831543
#         ],
#         [
#           -4.727778911590576
#         ],
#         [
#           -18.65475082397461
#         ],
#         [
#           -1.6789541244506836
#         ],
#         [
#           14.21212100982666
#         ],
#         [
#           4.164368152618408
#         ],
#         [
#           -10.573883056640625
#         ],
#         [
#           -4.93971586227417
#         ]
#       ],
#       [
#         [
#           -7.350412368774414
#         ],
#         [
#           -1.2212218046188354
#         ],
#         [
#           -0.07870417088270187
#         ],
#         [
#           -9.223077774047852
#         ],
#         [
#           1.7289559841156006
#         ],
#         [
#           -1.2825498580932617
#         ],
#         [
#           11.595855712890625
#         ],
#         [
#           2.6802620887756348
#         ],
#         [
#           -2.4769418239593506
#         ]
#       ]
#     ],
#     "final_loss": [
#       0.18372195274829473,
#       0.42402907499246956,
#       0.4674130108501502,
#       0.12179089146403355,
#       0.03680234199948762,
#       0.026212270091855228,
#       0.326429944314393,
#       0.2519309873059865,
#       0.25207173790430737,
#       0.22481433640766735,
#       0.27360906517629724,
#       0.2352076251092119,
#       0.40466381213794245,
#       0.3044501447015455,
#       0.25765153595096324,
#       0.4806103523203702,
#       0.27015225419529365,
#       0.20087502128275236,
#       0.3763018616501861,
#       0.2940192386566549,
#       0.5006288436232782,
#       2.7469514144361526,
#       0.3031946684754403,
#       0.11560081114645948,
#       0.2238754762300279
#     ],
#     "l2_gmm": [
#       0.13282912511310274,
#       0.28710861297970736,
#       0.31768034206834567,
#       0.1613908083596436,
#       0.09750287202273789,
#       0.0537692005242113,
#       0.7231541728095693,
#       0.164866070482606,
#       0.23116091895844532,
#       0.13057242263688212,
#       0.37148569272368837,
#       0.31086889290397113,
#       0.7231637564421914,
#       0.2030265432977359,
#       0.2719585855906111,
#       0.31835112092645335,
#       0.29422170509832635,
#       0.5038676164920666,
#       0.8416675160169181,
#       0.11566072583157111,
#       0.3089947479333973,
#       0.8007407352981074,
#       0.4315120201318565,
#       0.6328092314101645,
#       0.49913645758299796
#     ],
#     "l2_x": [
#       23.63296890258789,
#       21.547513961791992,
#       17.91318130493164,
#       25.61564064025879,
#       5.9219207763671875,
#       5.059595584869385,
#       12.182522773742676,
#       6.245851993560791,
#       8.65285873413086,
#       18.49824333190918,
#       6.874916076660156,
#       18.459089279174805,
#       16.971172332763672,
#       14.025564193725586,
#       42.722347259521484,
#       11.252840042114258,
#       16.088886260986328,
#       7.389485836029053,
#       13.128752708435059,
#       44.45333480834961,
#       17.123634338378906,
#       6.413752555847168,
#       5.439971923828125,
#       24.90561866760254,
#       10.158223152160645
#     ],
#     "times": [
#       187.07825827598572,
#       186.4305214881897,
#       188.63831067085266,
#       186.1335084438324,
#       179.98605608940125,
#       185.08248686790466,
#       194.63612461090088,
#       198.74575877189636,
#       187.55601286888123,
#       188.41356754302979,
#       188.1889796257019,
#       187.72968792915344,
#       188.36137890815735,
#       187.89114475250244,
#       188.80370664596558,
#       190.77033138275146,
#       189.97287821769714,
#       192.11608505249023,
#       189.8867962360382,
#       185.79658317565918,
#       186.32191848754883,
#       185.58427667617798,
#       184.34628033638,
#       185.3009045124054,
#       184.86836075782776
#     ]
#   },
#   "LGD-CM": {
#     "x_pred": [
#       [
#         [
#           -4.856094837188721
#         ],
#         [
#           1.4625204801559448
#         ],
#         [
#           0.4500931203365326
#         ],
#         [
#           -9.560831069946289
#         ],
#         [
#           1.5647178888320923
#         ],
#         [
#           1.2232273817062378
#         ],
#         [
#           2.3276004791259766
#         ],
#         [
#           -0.41247129440307617
#         ],
#         [
#           -5.7667927742004395
#         ]
#       ],
#       [
#         [
#           -4.91056489944458
#         ],
#         [
#           1.381649374961853
#         ],
#         [
#           0.3721340298652649
#         ],
#         [
#           -9.553133964538574
#         ],
#         [
#           1.5000988245010376
#         ],
#         [
#           1.1460586786270142
#         ],
#         [
#           2.4008002281188965
#         ],
#         [
#           -0.39104825258255005
#         ],
#         [
#           -5.8639631271362305
#         ]
#       ],
#       [
#         [
#           -4.7086501121521
#         ],
#         [
#           1.657345175743103
#         ],
#         [
#           0.42274224758148193
#         ],
#         [
#           -9.703625679016113
#         ],
#         [
#           1.8047747611999512
#         ],
#         [
#           1.3552881479263306
#         ],
#         [
#           2.354203462600708
#         ],
#         [
#           -0.42371517419815063
#         ],
#         [
#           -5.67739200592041
#         ]
#       ],
#       [
#         [
#           -4.800114631652832
#         ],
#         [
#           1.5861183404922485
#         ],
#         [
#           0.4220632016658783
#         ],
#         [
#           -9.64675235748291
#         ],
#         [
#           1.7010787725448608
#         ],
#         [
#           1.2081300020217896
#         ],
#         [
#           2.445122003555298
#         ],
#         [
#           -0.41344553232192993
#         ],
#         [
#           -5.727502822875977
#         ]
#       ],
#       [
#         [
#           -4.847458839416504
#         ],
#         [
#           1.4160794019699097
#         ],
#         [
#           0.44118982553482056
#         ],
#         [
#           -9.51809310913086
#         ],
#         [
#           1.5937925577163696
#         ],
#         [
#           1.2094944715499878
#         ],
#         [
#           2.310749053955078
#         ],
#         [
#           -0.4424920976161957
#         ],
#         [
#           -5.822656154632568
#         ]
#       ],
#       [
#         [
#           -4.893193244934082
#         ],
#         [
#           1.4893743991851807
#         ],
#         [
#           0.4053502380847931
#         ],
#         [
#           -9.605024337768555
#         ],
#         [
#           1.5909905433654785
#         ],
#         [
#           1.1686986684799194
#         ],
#         [
#           2.4903042316436768
#         ],
#         [
#           -0.44351619482040405
#         ],
#         [
#           -5.811279773712158
#         ]
#       ],
#       [
#         [
#           -4.928205966949463
#         ],
#         [
#           1.413155436515808
#         ],
#         [
#           0.3405245244503021
#         ],
#         [
#           -9.552335739135742
#         ],
#         [
#           1.5189449787139893
#         ],
#         [
#           1.1638103723526
#         ],
#         [
#           2.3679091930389404
#         ],
#         [
#           -0.4388259947299957
#         ],
#         [
#           -5.822782516479492
#         ]
#       ],
#       [
#         [
#           -4.687257289886475
#         ],
#         [
#           3.535431146621704
#         ],
#         [
#           -1.4421237707138062
#         ],
#         [
#           -15.263544082641602
#         ],
#         [
#           3.343669891357422
#         ],
#         [
#           0.37409675121307373
#         ],
#         [
#           8.236451148986816
#         ],
#         [
#           0.18034444749355316
#         ],
#         [
#           -1.7624844312667847
#         ]
#       ],
#       [
#         [
#           -5.0264997482299805
#         ],
#         [
#           1.5359296798706055
#         ],
#         [
#           0.23334936797618866
#         ],
#         [
#           -9.67066764831543
#         ],
#         [
#           1.6782221794128418
#         ],
#         [
#           1.1814934015274048
#         ],
#         [
#           2.802356243133545
#         ],
#         [
#           -0.49378013610839844
#         ],
#         [
#           -5.818561553955078
#         ]
#       ],
#       [
#         [
#           -4.921677112579346
#         ],
#         [
#           1.584564208984375
#         ],
#         [
#           0.22487446665763855
#         ],
#         [
#           -9.709822654724121
#         ],
#         [
#           1.700503945350647
#         ],
#         [
#           1.1513391733169556
#         ],
#         [
#           2.5233335494995117
#         ],
#         [
#           -0.42060989141464233
#         ],
#         [
#           -5.686213970184326
#         ]
#       ],
#       [
#         [
#           -4.771594524383545
#         ],
#         [
#           1.5685259103775024
#         ],
#         [
#           0.508869469165802
#         ],
#         [
#           -9.562230110168457
#         ],
#         [
#           1.6680439710617065
#         ],
#         [
#           1.2796027660369873
#         ],
#         [
#           2.352109432220459
#         ],
#         [
#           -0.46003758907318115
#         ],
#         [
#           -5.806147575378418
#         ]
#       ],
#       [
#         [
#           -4.677749156951904
#         ],
#         [
#           1.704241394996643
#         ],
#         [
#           0.5072488784790039
#         ],
#         [
#           -9.61740493774414
#         ],
#         [
#           1.8774456977844238
#         ],
#         [
#           1.4134498834609985
#         ],
#         [
#           2.251955509185791
#         ],
#         [
#           -0.528907299041748
#         ],
#         [
#           -5.625930309295654
#         ]
#       ],
#       [
#         [
#           -4.816075801849365
#         ],
#         [
#           1.5032708644866943
#         ],
#         [
#           0.4905926287174225
#         ],
#         [
#           -9.538118362426758
#         ],
#         [
#           1.6349709033966064
#         ],
#         [
#           1.2138630151748657
#         ],
#         [
#           2.368926763534546
#         ],
#         [
#           -0.418928861618042
#         ],
#         [
#           -5.872306823730469
#         ]
#       ],
#       [
#         [
#           -4.685406684875488
#         ],
#         [
#           1.5779173374176025
#         ],
#         [
#           0.6046409606933594
#         ],
#         [
#           -9.502166748046875
#         ],
#         [
#           1.6800849437713623
#         ],
#         [
#           1.346439242362976
#         ],
#         [
#           2.266623020172119
#         ],
#         [
#           -0.4760502278804779
#         ],
#         [
#           -5.74048376083374
#         ]
#       ],
#       [
#         [
#           -4.6715803146362305
#         ],
#         [
#           1.805720329284668
#         ],
#         [
#           0.485983669757843
#         ],
#         [
#           -9.59883975982666
#         ],
#         [
#           2.060988426208496
#         ],
#         [
#           1.460923433303833
#         ],
#         [
#           2.297154426574707
#         ],
#         [
#           -0.5389128923416138
#         ],
#         [
#           -5.635123252868652
#         ]
#       ],
#       [
#         [
#           -4.890048980712891
#         ],
#         [
#           1.5795326232910156
#         ],
#         [
#           0.26415345072746277
#         ],
#         [
#           -9.672061920166016
#         ],
#         [
#           1.7737163305282593
#         ],
#         [
#           1.2396962642669678
#         ],
#         [
#           2.407545566558838
#         ],
#         [
#           -0.45075511932373047
#         ],
#         [
#           -5.702228546142578
#         ]
#       ],
#       [
#         [
#           -4.681034088134766
#         ],
#         [
#           1.6096546649932861
#         ],
#         [
#           0.5986550450325012
#         ],
#         [
#           -9.490914344787598
#         ],
#         [
#           1.699908971786499
#         ],
#         [
#           1.3087142705917358
#         ],
#         [
#           2.2796790599823
#         ],
#         [
#           -0.5004069209098816
#         ],
#         [
#           -5.7540483474731445
#         ]
#       ],
#       [
#         [
#           -4.7694993019104
#         ],
#         [
#           1.7201852798461914
#         ],
#         [
#           0.36613738536834717
#         ],
#         [
#           -9.701086044311523
#         ],
#         [
#           1.8950012922286987
#         ],
#         [
#           1.4195713996887207
#         ],
#         [
#           2.3707199096679688
#         ],
#         [
#           -0.5154126882553101
#         ],
#         [
#           -5.6273980140686035
#         ]
#       ],
#       [
#         [
#           -5.0438337326049805
#         ],
#         [
#           1.3673714399337769
#         ],
#         [
#           0.3083863854408264
#         ],
#         [
#           -9.552433967590332
#         ],
#         [
#           1.47811758518219
#         ],
#         [
#           1.0593786239624023
#         ],
#         [
#           2.4997668266296387
#         ],
#         [
#           -0.44251254200935364
#         ],
#         [
#           -5.848386764526367
#         ]
#       ],
#       [
#         [
#           -4.788575172424316
#         ],
#         [
#           1.4763753414154053
#         ],
#         [
#           0.5353096127510071
#         ],
#         [
#           -9.471680641174316
#         ],
#         [
#           1.5810447931289673
#         ],
#         [
#           1.2273229360580444
#         ],
#         [
#           2.2431318759918213
#         ],
#         [
#           -0.47651368379592896
#         ],
#         [
#           -5.80408239364624
#         ]
#       ],
#       [
#         [
#           -4.589641094207764
#         ],
#         [
#           1.660406470298767
#         ],
#         [
#           0.6434038877487183
#         ],
#         [
#           -9.50139045715332
#         ],
#         [
#           1.8447898626327515
#         ],
#         [
#           1.381525993347168
#         ],
#         [
#           2.174748182296753
#         ],
#         [
#           -0.4626762568950653
#         ],
#         [
#           -5.789786338806152
#         ]
#       ],
#       [
#         [
#           -4.401574611663818
#         ],
#         [
#           1.8034899234771729
#         ],
#         [
#           0.7958076000213623
#         ],
#         [
#           -9.531121253967285
#         ],
#         [
#           1.997352957725525
#         ],
#         [
#           1.499567985534668
#         ],
#         [
#           2.191946506500244
#         ],
#         [
#           -0.4630928635597229
#         ],
#         [
#           -5.743812084197998
#         ]
#       ],
#       [
#         [
#           -4.814119815826416
#         ],
#         [
#           1.4946376085281372
#         ],
#         [
#           0.5338683724403381
#         ],
#         [
#           -9.415471076965332
#         ],
#         [
#           1.6464661359786987
#         ],
#         [
#           1.1677181720733643
#         ],
#         [
#           2.3329274654388428
#         ],
#         [
#           -0.46483805775642395
#         ],
#         [
#           -5.881463050842285
#         ]
#       ],
#       [
#         [
#           -4.710640907287598
#         ],
#         [
#           1.6107271909713745
#         ],
#         [
#           0.5922141075134277
#         ],
#         [
#           -9.518034934997559
#         ],
#         [
#           1.7222553491592407
#         ],
#         [
#           1.3013248443603516
#         ],
#         [
#           2.196092128753662
#         ],
#         [
#           -0.46606242656707764
#         ],
#         [
#           -5.799437999725342
#         ]
#       ],
#       [
#         [
#           -4.899129867553711
#         ],
#         [
#           1.4318453073501587
#         ],
#         [
#           0.3974178433418274
#         ],
#         [
#           -9.589913368225098
#         ],
#         [
#           1.5016087293624878
#         ],
#         [
#           1.1209994554519653
#         ],
#         [
#           2.4289019107818604
#         ],
#         [
#           -0.38323044776916504
#         ],
#         [
#           -5.795375823974609
#         ]
#       ]
#     ],
#     "final_loss": [
#       0.01718493434594537,
#       0.03089005339289308,
#       0.02449004724901327,
#       0.04340179870859595,
#       0.038812085471045865,
#       0.02186407973404636,
#       0.03680040345187274,
#       0.0965140445110535,
#       0.019610705287461005,
#       0.031223477716542103,
#       0.033575486496224105,
#       0.031100436706914536,
#       0.046253436224410915,
#       0.06359127928393526,
#       0.06771870846410222,
#       0.01638339353405227,
#       0.04302620153904879,
#       0.03784794038128281,
#       0.03522030715345981,
#       0.04525165969503142,
#       0.04049530695310288,
#       0.06096352482631273,
#       0.035269190515203164,
#       0.017937527550744292,
#       0.02819595454375179
#     ],
#     "l2_gmm": [
#       0.2831849021168843,
#       0.2749212503803108,
#       0.23543159924929402,
#       0.28899880469360284,
#       0.2607802912639007,
#       0.28323788192013233,
#       0.28410864344177034,
#       0.12893350225779165,
#       0.32168681026034274,
#       0.29384708680912486,
#       0.27777309801631633,
#       0.2605533044057732,
#       0.261155630001239,
#       0.33015697329036675,
#       0.265547262222698,
#       0.24082922766772352,
#       0.3471637749772537,
#       0.23869133218657523,
#       0.2979723075920745,
#       0.3023067980624734,
#       0.257697136107081,
#       0.27067011449629763,
#       0.33970971523123605,
#       0.2501010929518177,
#       0.30336571247653865
#     ],
#     "l2_x": [
#       4.194355010986328,
#       4.232630729675293,
#       4.007412433624268,
#       3.9987199306488037,
#       4.234668254852295,
#       4.0826215744018555,
#       4.213802337646484,
#       6.737830638885498,
#       3.916778326034546,
#       3.9157650470733643,
#       4.144870281219482,
#       4.0421013832092285,
#       4.186318397521973,
#       4.199299335479736,
#       3.9830682277679443,
#       3.994849920272827,
#       4.166233062744141,
#       3.9586071968078613,
#       4.168520927429199,
#       4.259287357330322,
#       4.209263801574707,
#       4.163454055786133,
#       4.227630138397217,
#       4.208370685577393,
#       4.148625373840332
#     ],
#     "times": [
#       68.49598383903503,
#       69.85892915725708,
#       67.6859073638916,
#       68.64116096496582,
#       69.00757670402527,
#       68.40800833702087,
#       69.16082215309143,
#       69.01811599731445,
#       68.8460373878479,
#       68.92287635803223,
#       70.54080295562744,
#       70.85620141029358,
#       69.40689444541931,
#       68.28127360343933,
#       67.64770174026489,
#       68.46227169036865,
#       67.47331595420837,
#       68.58709359169006,
#       69.96111416816711,
#       68.3262939453125,
#       70.07432723045349,
#       70.01156783103943,
#       70.16201162338257,
#       68.84170413017273,
#       69.00346612930298
#     ]
#   },
#   "D-Flow": {
#     "x_pred": [
#       [
#         [
#           -5.341897487640381
#         ],
#         [
#           0.3980770409107208
#         ],
#         [
#           -7.1617960929870605
#         ],
#         [
#           -6.437186241149902
#         ],
#         [
#           0.9167155623435974
#         ],
#         [
#           -0.9771731495857239
#         ],
#         [
#           -4.016726970672607
#         ],
#         [
#           -3.498678684234619
#         ],
#         [
#           -9.13866901397705
#         ]
#       ],
#       [
#         [
#           14.117673873901367
#         ],
#         [
#           -4.209691524505615
#         ],
#         [
#           2.422865867614746
#         ],
#         [
#           -12.808545112609863
#         ],
#         [
#           7.188582897186279
#         ],
#         [
#           -6.194633483886719
#         ],
#         [
#           1.6884970664978027
#         ],
#         [
#           20.921213150024414
#         ],
#         [
#           -2.46303129196167
#         ]
#       ],
#       [
#         [
#           -5.44405460357666
#         ],
#         [
#           -0.09677892923355103
#         ],
#         [
#           -2.2857024669647217
#         ],
#         [
#           -7.007502555847168
#         ],
#         [
#           3.6070339679718018
#         ],
#         [
#           3.1147868633270264
#         ],
#         [
#           -7.333045482635498
#         ],
#         [
#           -4.869391441345215
#         ],
#         [
#           -6.737395286560059
#         ]
#       ],
#       [
#         [
#           -2.4719491004943848
#         ],
#         [
#           -2.4316375255584717
#         ],
#         [
#           -4.27227258682251
#         ],
#         [
#           -1.1718566417694092
#         ],
#         [
#           1.5547881126403809
#         ],
#         [
#           1.483256459236145
#         ],
#         [
#           -0.8979030847549438
#         ],
#         [
#           -6.619647979736328
#         ],
#         [
#           -7.7149152755737305
#         ]
#       ],
#       [
#         [
#           -9.435734748840332
#         ],
#         [
#           1.327569603919983
#         ],
#         [
#           -2.834383487701416
#         ],
#         [
#           -9.462270736694336
#         ],
#         [
#           4.397481918334961
#         ],
#         [
#           6.9196929931640625
#         ],
#         [
#           -6.244621276855469
#         ],
#         [
#           -5.612332344055176
#         ],
#         [
#           -3.2517285346984863
#         ]
#       ],
#       [
#         [
#           -2.6421689987182617
#         ],
#         [
#           5.588095664978027
#         ],
#         [
#           -2.5074188709259033
#         ],
#         [
#           -2.0067713260650635
#         ],
#         [
#           2.7522470951080322
#         ],
#         [
#           -1.6384614706039429
#         ],
#         [
#           -9.21062183380127
#         ],
#         [
#           0.32814252376556396
#         ],
#         [
#           -4.680306911468506
#         ]
#       ],
#       [
#         [
#           -3.4849939346313477
#         ],
#         [
#           -1.8329356908798218
#         ],
#         [
#           -0.6733699440956116
#         ],
#         [
#           -1.2169227600097656
#         ],
#         [
#           0.3662727475166321
#         ],
#         [
#           0.32417285442352295
#         ],
#         [
#           3.2481162548065186
#         ],
#         [
#           -5.336752891540527
#         ],
#         [
#           -8.587455749511719
#         ]
#       ],
#       [
#         [
#           -5.077080726623535
#         ],
#         [
#           3.1681745052337646
#         ],
#         [
#           -2.7904388904571533
#         ],
#         [
#           -4.4088969230651855
#         ],
#         [
#           1.2234982252120972
#         ],
#         [
#           -2.310001850128174
#         ],
#         [
#           -4.234723091125488
#         ],
#         [
#           -4.0519819259643555
#         ],
#         [
#           -5.702378273010254
#         ]
#       ],
#       [
#         [
#           -6.476738452911377
#         ],
#         [
#           2.1228954792022705
#         ],
#         [
#           4.884425163269043
#         ],
#         [
#           -6.51394510269165
#         ],
#         [
#           6.699233055114746
#         ],
#         [
#           -0.520738422870636
#         ],
#         [
#           5.35774040222168
#         ],
#         [
#           -0.9544357657432556
#         ],
#         [
#           -8.351152420043945
#         ]
#       ],
#       [
#         [
#           -4.255502223968506
#         ],
#         [
#           -0.6894490122795105
#         ],
#         [
#           2.028261661529541
#         ],
#         [
#           -11.6393404006958
#         ],
#         [
#           4.993553638458252
#         ],
#         [
#           -6.8361101150512695
#         ],
#         [
#           5.732646942138672
#         ],
#         [
#           -1.4309775829315186
#         ],
#         [
#           0.8306836485862732
#         ]
#       ],
#       [
#         [
#           -3.667788028717041
#         ],
#         [
#           5.49351167678833
#         ],
#         [
#           -1.994429111480713
#         ],
#         [
#           -9.957551956176758
#         ],
#         [
#           2.8207614421844482
#         ],
#         [
#           0.9236981868743896
#         ],
#         [
#           -5.708734512329102
#         ],
#         [
#           3.1091244220733643
#         ],
#         [
#           -6.94851016998291
#         ]
#       ],
#       [
#         [
#           -2.908501386642456
#         ],
#         [
#           -3.523136615753174
#         ],
#         [
#           0.5011176466941833
#         ],
#         [
#           -9.715752601623535
#         ],
#         [
#           5.0998029708862305
#         ],
#         [
#           -2.351494789123535
#         ],
#         [
#           7.580819129943848
#         ],
#         [
#           7.03702974319458
#         ],
#         [
#           -5.441685199737549
#         ]
#       ],
#       [
#         [
#           11.372010231018066
#         ],
#         [
#           -17.090124130249023
#         ],
#         [
#           10.262100219726562
#         ],
#         [
#           -8.3080415725708
#         ],
#         [
#           -4.430649280548096
#         ],
#         [
#           4.504216194152832
#         ],
#         [
#           7.6770782470703125
#         ],
#         [
#           16.84316062927246
#         ],
#         [
#           7.491851329803467
#         ]
#       ],
#       [
#         [
#           -1.7439395189285278
#         ],
#         [
#           1.557751178741455
#         ],
#         [
#           -2.8178164958953857
#         ],
#         [
#           -3.9745829105377197
#         ],
#         [
#           3.0515589714050293
#         ],
#         [
#           1.513414978981018
#         ],
#         [
#           -4.44929313659668
#         ],
#         [
#           -3.5147745609283447
#         ],
#         [
#           -6.948858737945557
#         ]
#       ],
#       [
#         [
#           -6.394197463989258
#         ],
#         [
#           2.97912859916687
#         ],
#         [
#           -4.257915496826172
#         ],
#         [
#           -14.503668785095215
#         ],
#         [
#           1.9930018186569214
#         ],
#         [
#           0.09223897755146027
#         ],
#         [
#           4.632680892944336
#         ],
#         [
#           0.34687918424606323
#         ],
#         [
#           -3.7163636684417725
#         ]
#       ],
#       [
#         [
#           -6.079068660736084
#         ],
#         [
#           6.163812637329102
#         ],
#         [
#           -3.547244071960449
#         ],
#         [
#           -14.902235984802246
#         ],
#         [
#           3.7401199340820312
#         ],
#         [
#           -1.3217473030090332
#         ],
#         [
#           4.360175132751465
#         ],
#         [
#           2.0316171646118164
#         ],
#         [
#           -4.085818290710449
#         ]
#       ],
#       [
#         [
#           -7.278700351715088
#         ],
#         [
#           3.7258505821228027
#         ],
#         [
#           -1.5746433734893799
#         ],
#         [
#           -3.658212900161743
#         ],
#         [
#           2.4810843467712402
#         ],
#         [
#           2.412169933319092
#         ],
#         [
#           -2.617412805557251
#         ],
#         [
#           -4.397113800048828
#         ],
#         [
#           -3.4485201835632324
#         ]
#       ],
#       [
#         [
#           -5.896267414093018
#         ],
#         [
#           3.963686227798462
#         ],
#         [
#           0.9120476841926575
#         ],
#         [
#           -12.939231872558594
#         ],
#         [
#           4.684136390686035
#         ],
#         [
#           -3.575986862182617
#         ],
#         [
#           2.4731831550598145
#         ],
#         [
#           -1.4442692995071411
#         ],
#         [
#           -7.633181571960449
#         ]
#       ],
#       [
#         [
#           -5.443730354309082
#         ],
#         [
#           -1.0101230144500732
#         ],
#         [
#           1.4397205114364624
#         ],
#         [
#           -5.590760231018066
#         ],
#         [
#           1.3779078722000122
#         ],
#         [
#           2.1722824573516846
#         ],
#         [
#           -0.2053566724061966
#         ],
#         [
#           -0.5812452435493469
#         ],
#         [
#           -9.042417526245117
#         ]
#       ],
#       [
#         [
#           4.557478904724121
#         ],
#         [
#           10.860708236694336
#         ],
#         [
#           34.85298538208008
#         ],
#         [
#           -38.631038665771484
#         ],
#         [
#           8.49380874633789
#         ],
#         [
#           13.612896919250488
#         ],
#         [
#           -7.314835548400879
#         ],
#         [
#           16.557575225830078
#         ],
#         [
#           1.4075512886047363
#         ]
#       ],
#       [
#         [
#           -3.8396081924438477
#         ],
#         [
#           1.054593563079834
#         ],
#         [
#           -5.329183101654053
#         ],
#         [
#           -5.737791061401367
#         ],
#         [
#           2.348756790161133
#         ],
#         [
#           0.6133279800415039
#         ],
#         [
#           -6.284603118896484
#         ],
#         [
#           -4.203170299530029
#         ],
#         [
#           -4.868609428405762
#         ]
#       ],
#       [
#         [
#           10.84033489227295
#         ],
#         [
#           -5.027507781982422
#         ],
#         [
#           4.797849178314209
#         ],
#         [
#           -12.153846740722656
#         ],
#         [
#           6.747082233428955
#         ],
#         [
#           -8.550877571105957
#         ],
#         [
#           4.449919700622559
#         ],
#         [
#           8.218451499938965
#         ],
#         [
#           -0.036221668124198914
#         ]
#       ],
#       [
#         [
#           -0.7756141424179077
#         ],
#         [
#           7.485759735107422
#         ],
#         [
#           -1.9751957654953003
#         ],
#         [
#           -18.422447204589844
#         ],
#         [
#           8.170207023620605
#         ],
#         [
#           6.344823837280273
#         ],
#         [
#           2.316131591796875
#         ],
#         [
#           -1.0082488059997559
#         ],
#         [
#           -2.4829251766204834
#         ]
#       ],
#       [
#         [
#           -2.156101942062378
#         ],
#         [
#           8.422566413879395
#         ],
#         [
#           5.967549800872803
#         ],
#         [
#           -10.165979385375977
#         ],
#         [
#           1.270574927330017
#         ],
#         [
#           1.2894115447998047
#         ],
#         [
#           -5.078229904174805
#         ],
#         [
#           -9.343754768371582
#         ],
#         [
#           -1.8289591073989868
#         ]
#       ],
#       [
#         [
#           16.643625259399414
#         ],
#         [
#           -12.68114185333252
#         ],
#         [
#           13.805675506591797
#         ],
#         [
#           -5.185751914978027
#         ],
#         [
#           7.168396472930908
#         ],
#         [
#           -6.258316993713379
#         ],
#         [
#           7.140445232391357
#         ],
#         [
#           21.649261474609375
#         ],
#         [
#           -4.066987037658691
#         ]
#       ]
#     ],
#     "final_loss": [
#       0.7314182306156836,
#       0.5102700592226119,
#       0.6387986277011573,
#       0.8896545936953841,
#       0.74982674619261,
#       0.7263726416967788,
#       0.927216801872933,
#       1.0736651373852126,
#       0.4393051455473125,
#       0.9426400634340579,
#       0.9609891573212441,
#       1.1723504403099376,
#       0.6762252329977381,
#       0.8537186617507269,
#       0.5135180726581021,
#       0.4891213130612613,
#       0.9126552558282541,
#       0.4429898364068241,
#       0.909967003447671,
#       0.40145413928832907,
#       0.9648134229928518,
#       0.5205122542401148,
#       0.5975130025464912,
#       1.0139983739012566,
#       0.5381648168075612
#     ],
#     "l2_gmm": [
#       0.11669324924507762,
#       0.2078778066574134,
#       0.10161116915355822,
#       0.10865438280654094,
#       0.1744450876546051,
#       0.11050846434391759,
#       0.14613373361761028,
#       0.1614969834832385,
#       0.13337990536583502,
#       0.15988479465296768,
#       0.12883467296597384,
#       0.17109415546534867,
#       0.11606949755386342,
#       0.16017021721838598,
#       0.2729100128831016,
#       0.15479983010145323,
#       0.18390313937162484,
#       0.13171967234963589,
#       0.1534025021966734,
#       0.5045360204897359,
#       0.14650155290150457,
#       0.16952034979595643,
#       0.1709762598086383,
#       0.16291629526371254,
#       0.14068003349987507
#     ],
#     "l2_x": [
#       13.84875774383545,
#       30.804414749145508,
#       14.231098175048828,
#       15.217469215393066,
#       15.111543655395508,
#       17.0300235748291,
#       13.096055030822754,
#       12.088460922241211,
#       8.917939186096191,
#       9.030378341674805,
#       12.009568214416504,
#       11.546026229858398,
#       36.096920013427734,
#       12.914285659790039,
#       6.438908576965332,
#       7.465529441833496,
#       11.72478199005127,
#       5.766223907470703,
#       10.1813383102417,
#       52.7744255065918,
#       13.761382102966309,
#       22.274127960205078,
#       13.260032653808594,
#       15.220759391784668,
#       38.46487045288086
#     ],
#     "times": [
#       3.094841957092285,
#       5.090984106063843,
#       10.209907054901123,
#       4.553769111633301,
#       6.368864059448242,
#       8.684529542922974,
#       11.602131128311157,
#       6.051315546035767,
#       4.511491060256958,
#       7.601519584655762,
#       5.250146389007568,
#       6.285449981689453,
#       8.352883338928223,
#       5.801598072052002,
#       5.441418647766113,
#       4.649417877197266,
#       5.345767259597778,
#       9.856778860092163,
#       13.195526361465454,
#       11.198225259780884,
#       7.6086344718933105,
#       19.99777364730835,
#       5.505170583724976,
#       4.518542528152466,
#       3.4480082988739014
#     ]
#   },
#   "meta": {
#     "n_attemp_optim": 25,
#     "nsamples_in_optim_for_mmd": 250,
#     "x_star": [
#       -4.5403666496276855,
#       2.7113683223724365,
#       0.5512744784355164,
#       -11.295047760009766,
#       3.0334978103637695,
#       -0.6915482878684998,
#       4.155081272125244,
#       -1.238537311553955,
#       -4.014704704284668
#     ]
#   }
# }